# Market Basket Analysis with Associo

A practical example of market basket analysis on a grocery store dataset.

We'll:
1. Generate a realistic grocery dataset
2. Find product associations
3. Analyze and interpret key metrics
4. Filter for actionable rules

In [ ]:
import polars as pl
import random
from associo import combinatorial_associations

## 1. Generate Sample Data

We simulate 500 grocery orders with common product bundles.

In [ ]:
random.seed(42)

# Define product bundles with probabilities
bundles = [
    (["bread", "butter", "milk"], 0.3),
    (["pasta", "tomato_sauce", "parmesan"], 0.2),
    (["chips", "salsa", "beer"], 0.15),
    (["coffee", "sugar", "cream"], 0.2),
    (["eggs", "bacon", "bread"], 0.15),
    (["rice", "soy_sauce"], 0.1),
    (["yogurt", "granola", "berries"], 0.12),
]

# Random standalone products
standalone = ["apples", "bananas", "water", "juice", "cookies", "cereal", "chicken", "cheese"]

orders = []
for order_id in range(1, 501):
    items = set()
    # Add bundles with probability
    for bundle, prob in bundles:
        if random.random() < prob:
            items.update(bundle)
    # Add 1-3 random items
    items.update(random.sample(standalone, k=random.randint(1, 3)))
    for item in items:
        orders.append({"product": item, "order_id": order_id})

df = pl.DataFrame(orders)
print(f"Total rows: {len(df)}")
print(f"Unique orders: {df['order_id'].n_unique()}")
print(f"Unique products: {df['product'].n_unique()}")
df.head(10)

## 2. Compute All Pairwise Associations

In [ ]:
result = combinatorial_associations(
    df,
    column_items="product",
    column_tid="order_id",
)

print(f"Total association rules: {len(result)}")
print(f"Columns: {len(result.columns)}")
result.select("lhs", "rhs", "lhs_rhs_count", "support", "confidence", "lift").head(5)

## 3. Top Rules by Lift

**Lift > 1** means the items appear together more often than expected by chance.

In [ ]:
top_by_lift = (
    result
    .filter(pl.col("lhs_rhs_count") >= 10)  # minimum support count
    .select("lhs", "rhs", "lhs_rhs_count", "support", "confidence", "lift", "conviction", "zhangs_metric")
    .sort("lift", descending=True)
    .head(15)
)

top_by_lift

## 4. Filtering for Actionable Rules

Good rules typically satisfy multiple criteria:
- **Support** > threshold (the rule applies to enough transactions)
- **Confidence** > threshold (the rule is reliable)
- **Lift** > 1 (the items are positively associated)
- **Zhang's metric** > 0 (confirmed positive association)

In [ ]:
actionable = (
    result
    .filter(
        (pl.col("support") > 0.05)
        & (pl.col("confidence") > 0.3)
        & (pl.col("lift") > 1.5)
        & (pl.col("zhangs_metric") > 0)
    )
    .select("lhs", "rhs", "support", "confidence", "lift", "conviction", "zhangs_metric", "jaccard")
    .sort("confidence", descending=True)
)

print(f"Actionable rules: {len(actionable)}")
actionable

## 5. Interpreting the Metrics

Let's pick one rule and interpret all the key metrics.

In [ ]:
# Pick the strongest rule
best_rule = actionable.head(1)
lhs = best_rule["lhs"][0]
rhs = best_rule["rhs"][0]

rule_detail = (
    result
    .filter((pl.col("lhs") == lhs) & (pl.col("rhs") == rhs))
)

metrics_of_interest = [
    "lhs", "rhs", "lhs_rhs_count",
    "support", "confidence", "lift", "leverage",
    "conviction", "zhangs_metric", "jaccard", "cosine",
    "odds_ratio", "phi_coefficient", "mutual_information",
    "added_value", "certainty_factor", "relative_risk",
]

print(f"Rule: {lhs} → {rhs}")
print("=" * 40)
row = rule_detail.select(metrics_of_interest).row(0, named=True)
for key, val in row.items():
    if key in ("lhs", "rhs"):
        continue
    print(f"  {key:>30s}: {val}")

## 6. Comparing Symmetric vs Asymmetric Metrics

Some metrics are symmetric (`A→B == B→A`), while others are directional.

In [ ]:
# Compare A→B vs B→A for the top rule
pair = (
    result
    .filter(
        ((pl.col("lhs") == lhs) & (pl.col("rhs") == rhs))
        | ((pl.col("lhs") == rhs) & (pl.col("rhs") == lhs))
    )
    .select(
        "lhs", "rhs",
        "confidence",        # asymmetric
        "reverse_confidence", # asymmetric
        "lift",              # symmetric
        "jaccard",           # symmetric
        "conviction",        # asymmetric
        "zhangs_metric",     # asymmetric
    )
)

pair